# 05A — RSNA Knee Multi-Plane Dense-Slice MIL — Fold 0 SAFE

Engineering-only patch of 05A. The scientific design remains unchanged. This version keeps cached images as CPU `uint8 [B,T,H,W]`, performs float/RGB conversion and mild study-consistent augmentation only on GPU, uses conservative DataLoader settings, monitors memory, cleans up between folds, and runs only pre-existing fold 0 in this fresh Kaggle kernel.

Gold labels remain audit-only and never affect gradients, stopping, epochs, folds, or hyperparameters.


In [ ]:
# 1. Imports, fixed scientific configuration, and RAM-safe debug controls
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
from time import monotonic
import copy, ctypes, gc, hashlib, json, math, os, random, re, shutil, warnings, zipfile

import numpy as np
import pandas as pd
import pydicom
from sklearn.metrics import roc_auc_score, average_precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18

try:
    import psutil
except ImportError:
    psutil = None

warnings.filterwarnings("ignore")

CONFIG = {
    "experiment": "05A_RSNA_Knee_MultiPlane_DenseSlice_MIL_FOLD0_SAFE",
    "seed": 111,
    "smoke_test": False,
    "preflight_only": False,
    "preflight_weak_studies": 256,
    "preflight_epochs": 3,
    "dev_num_folds": 1,
    "active_folds": [0],
    "require_existing_cache": False,
    "image_size": 224,
    "physical_crop_mm": 140.0,
    "n_slices_per_slot": 16,
    "slice_dropout": 0.10,
    "slots": ["SAG_FLUID", "SAG_NONFLUID", "COR_FLUID", "COR_NONFLUID", "AX_FLUID"],
    "backbone": "resnet18",
    "backbone_weights": None,
    "token_dim": 256,
    "attention_heads": 4,
    "encoder_lr": 1e-4,
    "head_lr": 3e-4,
    "weight_decay": 1e-4,
    "max_epochs": 15,
    "patience": 4,
    "study_batch_size": 2,
    "gradient_accumulation_steps": 1,
    "num_workers": 1,
    "persistent_workers": False,
    "prefetch_factor": 1,
    "pin_memory": False,
    "max_clean_rss_growth_per_epoch_gib": 1.00,
    "memory_guard_warmup_epochs": 3,
    "min_system_available_gib": 4.0,
    "grad_clip_norm": 1.0,
    "cache_dtype": "uint8",
    "augment": True,
    "rotate_deg": 8.0,
    "brightness_jitter": 0.10,
    "gamma_min": 0.90,
    "gamma_max": 1.10,
}
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SLOTS = CONFIG["slots"]
OUT = Path("/kaggle/working/rsna_05a_dense_mil_fold0")
CACHE_DIR = Path("/kaggle/working/rsna_05a_dense_mil/study_cache")
OUT.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUN_START = monotonic()
MEMORY_HISTORY = []

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def print_memory_stats(tag):
    row = {"tag": tag, "elapsed_seconds": monotonic() - RUN_START,
           "process_rss_gib": np.nan, "children_rss_gib": np.nan, "total_process_tree_rss_gib": np.nan,
           "system_total_gib": np.nan,
           "system_used_gib": np.nan, "system_free_gib": np.nan, "system_available_gib": np.nan,
           "cuda_allocated_gib": 0.0, "cuda_reserved_gib": 0.0, "cuda_max_allocated_gib": 0.0}
    if psutil is not None:
        vm = psutil.virtual_memory(); proc = psutil.Process(os.getpid())
        child_rss = sum(c.memory_info().rss for c in proc.children(recursive=True) if c.is_running()) / 2**30
        main_rss = proc.memory_info().rss / 2**30
        row.update({"process_rss_gib": main_rss, "children_rss_gib": child_rss,
                    "total_process_tree_rss_gib": main_rss + child_rss, "system_total_gib": vm.total / 2**30,
                    "system_used_gib": vm.used / 2**30, "system_free_gib": vm.free / 2**30,
                    "system_available_gib": vm.available / 2**30})
    if torch.cuda.is_available():
        row.update({"cuda_allocated_gib": torch.cuda.memory_allocated() / 2**30,
                    "cuda_reserved_gib": torch.cuda.memory_reserved() / 2**30,
                    "cuda_max_allocated_gib": torch.cuda.max_memory_allocated() / 2**30})
    MEMORY_HISTORY.append(row)
    pd.DataFrame(MEMORY_HISTORY).to_csv(OUT / "memory_history.csv", index=False)
    print("MEMORY", tag, {k: (round(v, 3) if isinstance(v, float) and np.isfinite(v) else v)
                           for k, v in row.items() if k not in {"tag", "elapsed_seconds"}})
    return row

def release_host_memory():
    """Release cyclic objects and ask glibc to return free arenas to the OS."""
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

seed_everything(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
AMP = DEVICE.type == "cuda"
assert 1 <= CONFIG["dev_num_folds"] <= 5
assert CONFIG["active_folds"] and all(0 <= int(f) < 5 for f in CONFIG["active_folds"])
print(json.dumps(CONFIG, indent=2)); print("device:", DEVICE, "GPUs:", GPU_NAMES)
print("RUN_STRATEGY: ONE_FOLD_PER_KAGGLE_KERNEL_V4")
print("BATCH_TRANSPORT: NUMPY_PICKLE_V3")


## 2. Fast, shallow path resolution

No recursive scan of `/kaggle/input` is performed. Competition paths are checked explicitly; attached artifacts are searched only at depth one and two.


In [ ]:
def shallow_candidates(name, max_depth=5):
    """Bounded artifact lookup; prunes huge DICOM trees and never rglob-scans all input."""
    root = Path("/kaggle/input")
    ans = []
    if not root.exists(): return ans
    for current, dirs, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.relative_to(root).parts)
        # Never descend into the competition's millions of nested DICOM paths.
        dirs[:] = [d for d in dirs if d not in {"train_series", "test_series"}]
        if depth >= max_depth: dirs[:] = []
        if name in files: ans.append(current_path / name)
    print(f"Candidates for {name}:", [str(p) for p in ans])
    return sorted(set(ans), key=lambda p: (len(p.parts), str(p)))

def find_named_directories(name, max_depth=8):
    """Find a directory itself without enumerating the thousands of cache files inside it."""
    root = Path("/kaggle/input"); found = []
    if not root.exists(): return found
    for current, dirs, _ in os.walk(root):
        current_path = Path(current); depth = len(current_path.relative_to(root).parts)
        dirs[:] = [d for d in dirs if d not in {"train_series", "test_series"}]
        if name in dirs:
            found.append(current_path / name)
            dirs.remove(name)  # record it, but never walk through thousands of .npz files
        if depth >= max_depth: dirs[:] = []
    return sorted(set(found), key=str)

def csv_has(path, columns):
    try: return set(columns).issubset(pd.read_csv(path, nrows=2).columns)
    except Exception: return False

def resolve_paths():
    comp_candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]
    comp_candidates += [p.parent for p in shallow_candidates("train.csv")]
    comp = None
    required = ["train.csv", "train_series.csv", "train_series", "test.csv", "test_series.csv", "test_series"]
    for root in comp_candidates:
        if all((root / x).exists() for x in required) and csv_has(root / "train.csv", ["StudyInstanceUID", "Report"] + LABELS):
            comp = root; break
    if comp is None: raise FileNotFoundError("Competition root not found at expected shallow Kaggle input paths")

    silver_hits = [p for p in shallow_candidates("L1_family_mean_RECOMMENDED.csv") if csv_has(p, ["StudyInstanceUID"] + LABELS)]
    manifest_hits = [p for p in shallow_candidates("baseline_series_manifest.csv") if csv_has(p, ["StudyInstanceUID", "report_group"])]
    if not silver_hits: raise FileNotFoundError("Attach L1_family_mean_RECOMMENDED.csv from notebook 02")
    if not manifest_hits: raise FileNotFoundError("Attach baseline_series_manifest.csv from notebook 03")
    fold_hits = [p for p in shallow_candidates("fold_assignments.csv") if csv_has(p, ["StudyInstanceUID", "fold"])]
    # Optional resume source: attach the saved output of an earlier failed 05A version.
    prior_cache_dirs = find_named_directories("study_cache", max_depth=8)
    for cache_manifest in shallow_candidates("weak_cache_manifest.csv", max_depth=8):
        candidate = cache_manifest.parent / "study_cache"
        if candidate.is_dir(): prior_cache_dirs.append(candidate)
    prior_cache_dirs = sorted(set(prior_cache_dirs), key=str)
    cache_inventory = []
    for d in prior_cache_dirs:
        weak_count = sum(1 for _ in os.scandir(d) if _.is_file() and _.name.startswith("weak_") and _.name.endswith(".npz"))
        gold_count = sum(1 for _ in os.scandir(d) if _.is_file() and _.name.startswith("gold_") and _.name.endswith(".npz"))
        cache_inventory.append({"directory": str(d), "weak_npz": weak_count, "gold_npz": gold_count})
    print("Detected study-cache directories:", cache_inventory)
    return {"competition": comp, "silver": silver_hits[0], "manifest": manifest_hits[0],
            "prior_folds": fold_hits[0] if fold_hits else None,
            "prior_cache_dirs": prior_cache_dirs, "cache_inventory": cache_inventory}

PATHS = resolve_paths()
for k, v in PATHS.items(): print(f"{k}: {v}")


In [ ]:
# 3. Load tables and enforce the weak/gold separation contract
COMP_ROOT = PATHS["competition"]
TRAIN_DICOM_ROOT = COMP_ROOT / "train_series"
train = pd.read_csv(COMP_ROOT / "train.csv", dtype={"StudyInstanceUID": str})
silver = pd.read_csv(PATHS["silver"], dtype={"StudyInstanceUID": str})
manifest = pd.read_csv(PATHS["manifest"], dtype={"StudyInstanceUID": str})

assert len(silver) == 4349 and silver.StudyInstanceUID.is_unique
assert all(c in silver.columns for c in LABELS) and not silver[LABELS].isna().any().any()
assert np.isfinite(silver[LABELS].to_numpy(float)).all()
gold = train.loc[train[LABELS].notna().all(axis=1), ["StudyInstanceUID"] + LABELS].copy()
assert len(gold) == 58
gold_ids, weak_ids = set(gold.StudyInstanceUID), set(silver.StudyInstanceUID)
assert gold_ids.isdisjoint(weak_ids)
assert set(train.StudyInstanceUID) == set(manifest.StudyInstanceUID)
assert all(f"{s}__SeriesInstanceUID" in manifest.columns for s in SLOTS)

weak_manifest = manifest[manifest.StudyInstanceUID.isin(weak_ids)].merge(
    silver, on="StudyInstanceUID", how="left", validate="one_to_one")
gold_manifest = gold[["StudyInstanceUID"]].merge(
    manifest, on="StudyInstanceUID", how="left", validate="one_to_one")
assert len(weak_manifest) == 4349 and len(gold_manifest) == 58

if CONFIG["smoke_test"]:
    weak_manifest = weak_manifest.iloc[:32].copy()
elif CONFIG["preflight_only"]:
    # Deterministic subset; same model/data path, short enough to validate repeated-epoch RSS first.
    weak_manifest = weak_manifest.iloc[:CONFIG["preflight_weak_studies"]].copy()
print("weak production count:", len(silver), "active:", len(weak_manifest), "gold audit:", len(gold_manifest),
      "preflight_only:", CONFIG["preflight_only"])


## 4. Geometry ordering, slice sampling, and physical preprocessing

Only IOP+IPP projected geometry is primary. `InstanceNumber` is the sole fallback. Files with unreadable headers can be skipped when at least one safe ordered stack remains; filenames and SOP UIDs never determine anatomical order.


In [ ]:
STATS = Counter()
DECODE_TRANSFER_SYNTAX_FAILURES = Counter()

def dcm_files(series_dir):
    try: return [Path(e.path) for e in os.scandir(series_dir) if e.is_file() and e.name.lower().endswith(".dcm")]
    except Exception: return []

def safe_float(x, default=np.nan):
    try: return float(x)
    except Exception: return default

def sort_dicom_series(series_dir):
    records = []
    for path in dcm_files(series_dir):
        try:
            ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
            iop, ipp = getattr(ds, "ImageOrientationPatient", None), getattr(ds, "ImagePositionPatient", None)
            position = np.nan
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                a, b = np.asarray(iop[:6], float), np.asarray(ipp[:3], float)
                normal = np.cross(a[:3], a[3:])
                if np.isfinite(a).all() and np.isfinite(b).all() and np.linalg.norm(normal) > 1e-8:
                    position = float(np.dot(b, normal))
            records.append((path, position, safe_float(getattr(ds, "InstanceNumber", np.nan))))
        except Exception:
            STATS["failed_headers"] += 1
    if not records: return []
    if all(np.isfinite(x[1]) for x in records):
        records.sort(key=lambda x: x[1]); STATS["geometry_sorted_series"] += 1
    elif all(np.isfinite(x[2]) for x in records):
        records.sort(key=lambda x: x[2]); STATS["instance_fallback_series"] += 1
    else:
        STATS["failed_series_order"] += 1; return []
    return [x[0] for x in records]

def sample_slice_indices(n, budget):
    if n <= 0: return np.empty(0, dtype=np.int32)
    if n <= budget: return np.arange(n, dtype=np.int32)
    return np.unique(np.rint(np.linspace(0, n - 1, budget)).astype(np.int32))

def physical_center_crop(arr, row_spacing, col_spacing, crop_mm):
    h, w = arr.shape
    ch = min(h, max(1, int(round(crop_mm / row_spacing))))
    cw = min(w, max(1, int(round(crop_mm / col_spacing))))
    y0, x0 = max(0, (h - ch) // 2), max(0, (w - cw) // 2)
    return arr[y0:y0 + ch, x0:x0 + cw]

def read_dicom_slice(path):
    transfer = "unknown"
    try:
        ds = pydicom.dcmread(str(path), force=True)
        transfer = str(getattr(getattr(ds, "file_meta", None), "TransferSyntaxUID", "unknown"))
        arr = np.asarray(ds.pixel_array, dtype=np.float32).squeeze()
        if arr.ndim != 2: raise ValueError(f"expected 2D slice, got {arr.shape}")
        arr = arr * safe_float(getattr(ds, "RescaleSlope", 1.0), 1.0) + safe_float(getattr(ds, "RescaleIntercept", 0.0), 0.0)
        finite = np.isfinite(arr)
        if not finite.any(): raise ValueError("all pixels non-finite")
        arr[~finite] = np.median(arr[finite])
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            arr = float(arr.max() + arr.min()) - arr
        lo, hi = np.percentile(arr, [1, 99])
        arr = np.zeros_like(arr) if hi <= lo + 1e-6 else np.clip((np.clip(arr, lo, hi) - lo) / (hi - lo), 0, 1)
        spacing = getattr(ds, "PixelSpacing", None)
        if spacing is None or len(spacing) < 2: raise ValueError("missing PixelSpacing")
        rs, cs = float(spacing[0]), float(spacing[1])
        if not (np.isfinite(rs) and np.isfinite(cs) and rs > 0 and cs > 0): raise ValueError("invalid PixelSpacing")
        arr = physical_center_crop(arr, rs, cs, CONFIG["physical_crop_mm"])
        t = torch.from_numpy(arr)[None, None].float()
        t = torch.nn.functional.interpolate(t, size=(CONFIG["image_size"], CONFIG["image_size"]),
                                            mode="bilinear", align_corners=False)[0, 0]
        return np.rint(t.clamp(0, 1).numpy() * 255).astype(np.uint8)
    except Exception:
        STATS["failed_pixel_decodes"] += 1
        DECODE_TRANSFER_SYNTAX_FAILURES[transfer] += 1
        return None


## 5. Reuse cache when present; rebuild only missing studies

The saved failed-version output does not contain the 11.9 GiB `study_cache/`. RAMFIX therefore validates and reuses every compatible local/attached `.npz`, but rebuilds a study from DICOM only when its cache file is genuinely absent. Incompatible cache files still fail loudly rather than being silently accepted.


In [ ]:
def validate_cache_file(path):
    try:
        with np.load(path) as z:
            required = {"images", "slot_ids", "slice_indices", "positions"}
            if not required.issubset(z.files): return None
            images, slots = z["images"], z["slot_ids"]
            indices, positions = z["slice_indices"], z["positions"]
            n = len(images)
            if images.dtype != np.uint8 or images.shape != (n, CONFIG["image_size"], CONFIG["image_size"]): return None
            if not (len(slots) == len(indices) == len(positions) == n): return None
            if n > len(SLOTS) * CONFIG["n_slices_per_slot"]: return None
            if n and (slots.min() < 0 or slots.max() >= len(SLOTS)): return None
            if n and (not np.isfinite(positions).all() or positions.min() < 0 or positions.max() > 1): return None
            return n
    except Exception:
        return None

def cache_study(row, split):
    uid = str(row["StudyInstanceUID"])
    local_path = CACHE_DIR / f"{split}_{uid}.npz"
    candidates = [(local_path, "local")]
    candidates += [(Path(d) / f"{split}_{uid}.npz", "attached") for d in PATHS["prior_cache_dirs"]]
    existing = [x for x in candidates if x[0].exists()]
    for candidate, source in existing:
        n_tokens = validate_cache_file(candidate)
        if n_tokens is None:
            raise RuntimeError(f"Incompatible 05A cache file: {candidate}")
        STATS[f"reused_{source}_cached_studies"] += 1
        return {"StudyInstanceUID": uid, "cache_path": str(candidate), "n_tokens": n_tokens}
    if CONFIG["require_existing_cache"]:
        raise FileNotFoundError(
            f"Missing cache for {split} study {uid}. Detected cache inventory: {PATHS['cache_inventory']}. "
            "Attach the saved output that actually contains rsna_05a_dense_mil/study_cache with "
            "weak_*.npz and gold_*.npz files; a manifest or artifacts ZIP alone is insufficient."
        )

    # Disabled by default; retained only for an explicitly requested future from-scratch run.
    images, slot_ids, slice_indices, positions = [], [], [], []
    for slot_id, slot in enumerate(SLOTS):
        series_uid = row.get(f"{slot}__SeriesInstanceUID", np.nan)
        if pd.isna(series_uid) or not str(series_uid).strip(): continue
        ordered = sort_dicom_series(TRAIN_DICOM_ROOT / uid / str(series_uid))
        for idx in sample_slice_indices(len(ordered), CONFIG["n_slices_per_slot"]):
            image = read_dicom_slice(ordered[int(idx)])
            if image is None: continue
            images.append(image); slot_ids.append(slot_id); slice_indices.append(int(idx))
            positions.append(float(idx / max(1, len(ordered) - 1)))
    arr = np.stack(images) if images else np.empty((0, CONFIG["image_size"], CONFIG["image_size"]), np.uint8)
    np.savez_compressed(local_path, images=arr, slot_ids=np.asarray(slot_ids, np.int8),
                        slice_indices=np.asarray(slice_indices, np.int16), positions=np.asarray(positions, np.float32))
    STATS["rebuilt_cached_studies"] += 1
    return {"StudyInstanceUID": uid, "cache_path": str(local_path), "n_tokens": len(images)}

def build_cache_manifest(manifest_df, split):
    rows = []
    for i, (_, row) in enumerate(manifest_df.iterrows(), 1):
        rows.append(cache_study(row, split))
        if i % 500 == 0: print(f"validated/reused {split}: {i}/{len(manifest_df)}")
    out = pd.DataFrame(rows)
    if (out.n_tokens == 0).any():
        print("WARNING: bias-only empty studies:", out.loc[out.n_tokens.eq(0), "StudyInstanceUID"].tolist())
    out.to_csv(OUT / f"{split}_cache_manifest.csv", index=False)
    return out

if not PATHS["prior_cache_dirs"] and not any(CACHE_DIR.glob("weak_*.npz")):
    free_gib = shutil.disk_usage("/kaggle/working").free / 2**30
    print(f"No reusable study cache detected; rebuilding missing cache once. Free /kaggle/working space: {free_gib:.2f} GiB")
    if free_gib < 14.0:
        raise RuntimeError(f"Insufficient working-disk space for the ~11.9 GiB cache: only {free_gib:.2f} GiB free")

weak_cache_manifest = build_cache_manifest(weak_manifest, "weak")
gold_cache_manifest = build_cache_manifest(gold_manifest, "gold")
CACHE_BYTES = sum(Path(p).stat().st_size for p in pd.concat([weak_cache_manifest, gold_cache_manifest]).cache_path)
print("referenced cache GiB:", CACHE_BYTES / 2**30)
print("average weak valid slices/study:", weak_cache_manifest.n_tokens.mean())
print("cache stats:", dict(STATS))
print_memory_stats("after cache setup")


## 6. RAM-safe variable-token dataset and persistent decompression worker

CPU batches remain compact `uint8 [B,T,H,W]`. One persistent worker owns and reuses the NumPy/ZIP decompression heap across epochs; this avoids the measured 1.4 GiB/epoch growth in the main process caused by repeatedly allocating compressed-cache buffers. Float normalization and RGB expansion still occur only after H2D.


In [ ]:
class CachedStudyDataset(Dataset):
    def __init__(self, cache_manifest, target_frame=None, training=False):
        self.rows = cache_manifest.reset_index(drop=True)
        self.targets = None if target_frame is None else target_frame.set_index("StudyInstanceUID")
        self.training = training
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        row = self.rows.iloc[i]; uid = str(row.StudyInstanceUID)
        with np.load(row.cache_path) as z:
            # npz members are already independent decompressed arrays; avoid a second full copy.
            images = z["images"]; slots = z["slot_ids"]
            indices = z["slice_indices"]; positions = z["positions"]
        if self.training and len(images) > 1 and CONFIG["slice_dropout"] > 0:
            keep = np.random.random(len(images)) >= CONFIG["slice_dropout"]
            if not keep.any(): keep[np.random.randint(len(images))] = True
            images, slots, indices, positions = images[keep], slots[keep], indices[keep], positions[keep]
        target = None if self.targets is None else self.targets.loc[uid, LABELS].to_numpy(np.float32)
        return {"uid": uid, "images": images, "slots": slots, "slice_indices": indices,
                "positions": positions, "target": target}

def collate_fn(items):
    # IMPORTANT: keep worker output as plain NumPy. Returning torch.Tensor objects from a
    # multiprocessing DataLoader uses torch shared-memory storage; on this Kaggle image those
    # mappings accumulated by ~1.42 GiB/epoch even after workers exited.
    b = len(items); max_t = max(1, max(len(x["images"]) for x in items)); h = CONFIG["image_size"]
    images = np.zeros((b, max_t, h, h), dtype=np.uint8)
    slots = np.zeros((b, max_t), dtype=np.int64)
    positions = np.zeros((b, max_t), dtype=np.float32)
    mask = np.zeros((b, max_t), dtype=np.bool_)
    slice_indices = np.full((b, max_t), -1, dtype=np.int64)
    for i, x in enumerate(items):
        n = len(x["images"])
        if n:
            images[i, :n] = x["images"]
            slots[i, :n] = x["slots"]
            positions[i, :n] = x["positions"]
            slice_indices[i, :n] = x["slice_indices"]
            mask[i, :n] = True
    targets = None if items[0]["target"] is None else np.stack([x["target"] for x in items]).astype(np.float32, copy=False)
    assert images.dtype == np.uint8 and images.ndim == 4
    return {"uids": [x["uid"] for x in items], "images": images, "slots": slots,
            "positions": positions, "slice_indices": slice_indices, "mask": mask, "targets": targets}

def worker_init_fn(worker_id):
    seed = (CONFIG["seed"] + worker_id + torch.initial_seed()) % (2**32 - 1)
    np.random.seed(seed); random.seed(seed)

def make_loader(cache_manifest, targets=None, training=False, shuffle=False):
    workers = 0 if CONFIG["smoke_test"] else CONFIG["num_workers"]
    kwargs = dict(dataset=CachedStudyDataset(cache_manifest, targets, training),
                  batch_size=CONFIG["study_batch_size"], shuffle=shuffle, num_workers=workers,
                  pin_memory=CONFIG["pin_memory"], persistent_workers=CONFIG["persistent_workers"] and workers > 0,
                  collate_fn=collate_fn, worker_init_fn=worker_init_fn, drop_last=False)
    if workers > 0: kwargs["prefetch_factor"] = CONFIG["prefetch_factor"]
    return DataLoader(**kwargs)


## 7. Unchanged ResNet18 + pathology-specific MIL, with GPU-only conversion and mild augmentation

The module/state-dict architecture is unchanged. Training augmentation uses one rotation, brightness, and gamma draw per study and applies it consistently to every slice. There are no flips and no destructive crop.


In [ ]:
class DenseSliceMIL(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None); backbone.fc = nn.Identity(); self.encoder = backbone
        d = CONFIG["token_dim"]
        self.slice_projection = nn.Sequential(nn.Linear(512, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(0.1))
        self.slot_embedding = nn.Embedding(len(SLOTS), d)
        self.position_embedding = nn.Sequential(nn.Linear(1, d), nn.GELU(), nn.Linear(d, d))
        self.queries = nn.Parameter(torch.randn(len(LABELS), d) / math.sqrt(d))
        self.attention = nn.MultiheadAttention(d, CONFIG["attention_heads"], batch_first=True, dropout=0.1)
        self.classifiers = nn.ModuleList([nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1)) for _ in LABELS])

    def augment_gray(self, gray):
        if not (self.training and CONFIG["augment"]): return gray
        b, t, _, h, w = gray.shape
        angle = (torch.rand(b, device=gray.device) * 2 - 1) * math.radians(CONFIG["rotate_deg"])
        cos, sin = torch.cos(angle), torch.sin(angle)
        theta = torch.zeros((b, 2, 3), device=gray.device, dtype=gray.dtype)
        theta[:, 0, 0] = cos; theta[:, 0, 1] = -sin
        theta[:, 1, 0] = sin; theta[:, 1, 1] = cos
        theta = theta.repeat_interleave(t, dim=0)
        flat = gray.reshape(b * t, 1, h, w)
        grid = F.affine_grid(theta, flat.shape, align_corners=False)
        gray = F.grid_sample(flat, grid, mode="bilinear", padding_mode="zeros", align_corners=False).reshape(b, t, 1, h, w)
        brightness = 1 + (torch.rand((b, 1, 1, 1, 1), device=gray.device) * 2 - 1) * CONFIG["brightness_jitter"]
        gamma = CONFIG["gamma_min"] + torch.rand((b, 1, 1, 1, 1), device=gray.device) * (CONFIG["gamma_max"] - CONFIG["gamma_min"])
        return (gray * brightness).clamp(0, 1).pow(gamma)

    def forward(self, images, slots, positions, mask, return_attention=False):
        assert images.dtype == torch.uint8 and images.ndim == 4
        b, t = mask.shape; d = CONFIG["token_dim"]
        # Compact uint8 has already crossed H2D. Float/RGB materialization happens only here on GPU.
        gray = images.float().div_(255.0).unsqueeze(2)
        gray = self.augment_gray(gray)
        rgb = gray.expand(-1, -1, 3, -1, -1)
        tokens = positions.new_zeros((b, t, d))
        if mask.any():
            valid_rgb = rgb[mask].contiguous()
            encoded = self.encoder(valid_rgb)
            projected = self.slice_projection(encoded)
            projected = projected + self.slot_embedding(slots[mask]) + self.position_embedding(positions[mask].unsqueeze(-1))
            tokens[mask] = projected.to(tokens.dtype)
        pooled = tokens.new_zeros((b, len(LABELS), d))
        weights = tokens.new_zeros((b, CONFIG["attention_heads"], len(LABELS), t))
        nonempty = mask.any(dim=1)
        if nonempty.any():
            q = self.queries[None].expand(int(nonempty.sum()), -1, -1)
            po, aw = self.attention(q, tokens[nonempty], tokens[nonempty], key_padding_mask=~mask[nonempty],
                                    need_weights=return_attention, average_attn_weights=False)
            pooled[nonempty] = po.to(pooled.dtype)
            if return_attention: weights[nonempty] = aw.to(weights.dtype)
        logits = torch.cat([head(pooled[:, j]).reshape(b, 1) for j, head in enumerate(self.classifiers)], dim=1)
        return (logits, weights) if return_attention else logits

def unwrap(model): return model.module if isinstance(model, nn.DataParallel) else model

probe = DenseSliceMIL()
PARAMS = sum(p.numel() for p in probe.parameters()); TRAINABLE_PARAMS = sum(p.numel() for p in probe.parameters() if p.requires_grad)
print("parameters:", PARAMS, "trainable:", TRAINABLE_PARAMS)
del probe; gc.collect()


## 8. Reuse existing folds, strengthen leakage checks, and audit scanner/acquisition balance

RAMFIX requires the previously saved fold assignment. Scanner metadata is audit-only and cannot alter the split or enter the model.


In [ ]:
def load_existing_folds(df):
    prior = PATHS["prior_folds"]
    if prior is None:
        raise FileNotFoundError("Attach the previous 05A/04 output containing fold_assignments.csv; RAMFIX will not create new folds")
    old = pd.read_csv(prior, dtype={"StudyInstanceUID": str})[["StudyInstanceUID", "fold"]]
    merged = df[["StudyInstanceUID"]].merge(old, on="StudyInstanceUID", how="left", validate="one_to_one")
    assert len(merged) == len(df) and merged.fold.notna().all()
    folds = merged.fold.astype(int).to_numpy()
    if CONFIG["preflight_only"]:
        assert set(folds).issubset(set(range(5))) and 0 in set(folds)
    else:
        assert set(folds) == set(range(5)), f"Expected existing folds 0..4, got {sorted(set(folds))}"
    ids = df.StudyInstanceUID.astype(str).to_numpy(); groups = df.report_group.astype(str).to_numpy()
    for fold in range(5):
        tr, va = folds != fold, folds == fold
        assert set(groups[tr]).isdisjoint(set(groups[va]))
        assert set(ids[tr]).isdisjoint(gold_ids)
        assert set(ids[va]).isdisjoint(gold_ids)
    print("Reused unchanged fold assignments:", prior)
    return folds

fold_ids = load_existing_folds(weak_manifest)
fold_assignment_df = pd.DataFrame({"StudyInstanceUID": weak_manifest.StudyInstanceUID,
                                    "report_group": weak_manifest.report_group, "fold": fold_ids})
fold_assignment_df.to_csv(OUT / "fold_assignments.csv", index=False)
ACTIVE_FOLDS = [0] if CONFIG["preflight_only"] else [int(f) for f in CONFIG["active_folds"]]

def finite_summary(series, fn="median"):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if not len(x): return np.nan
    return float(x.median() if fn == "median" else x.mean())

audit_rows = []
audit_source = weak_manifest.copy(); audit_source["fold"] = fold_ids
for slot in SLOTS:
    mcol = f"{slot}__Manufacturer"
    if mcol not in audit_source.columns: continue
    d = audit_source[["StudyInstanceUID", "fold", mcol] + [c for c in [
        f"{slot}__PixelSpacingMean", f"{slot}__SliceThickness", f"{slot}__FOVRow_mm", f"{slot}__FOVCol_mm",
        f"{slot}__MagneticFieldStrength", f"{slot}__Rows", f"{slot}__Columns"] if c in audit_source.columns]].copy()
    d["manufacturer"] = d[mcol].fillna("UNKNOWN").astype(str).replace("", "UNKNOWN")
    totals = d.groupby("fold").size().to_dict()
    for (fold, manufacturer), g in d.groupby(["fold", "manufacturer"], dropna=False):
        row = {"fold": int(fold), "slot": slot, "manufacturer": manufacturer, "n_studies": len(g),
               "manufacturer_fraction": len(g) / totals[int(fold)],
               "pixel_spacing_median": finite_summary(g.get(f"{slot}__PixelSpacingMean", pd.Series(dtype=float))),
               "slice_thickness_median": finite_summary(g.get(f"{slot}__SliceThickness", pd.Series(dtype=float))),
               "fov_row_median": finite_summary(g.get(f"{slot}__FOVRow_mm", pd.Series(dtype=float))),
               "fov_col_median": finite_summary(g.get(f"{slot}__FOVCol_mm", pd.Series(dtype=float))),
               "magnetic_field_strength_median": finite_summary(g.get(f"{slot}__MagneticFieldStrength", pd.Series(dtype=float))),
               "rows_median": finite_summary(g.get(f"{slot}__Rows", pd.Series(dtype=float))),
               "columns_median": finite_summary(g.get(f"{slot}__Columns", pd.Series(dtype=float))),
               "field_strength_status": "available" if f"{slot}__MagneticFieldStrength" in g.columns else "not_available_in_manifest"}
        audit_rows.append(row)
scanner_fold_audit = pd.DataFrame(audit_rows)
scanner_fold_audit.to_csv(OUT / "scanner_fold_audit.csv", index=False)
imbalance = scanner_fold_audit.groupby(["slot", "manufacturer"]).manufacturer_fraction.agg(lambda x: x.max() - x.min())
LARGEST_MANUFACTURER_IMBALANCE = float(imbalance.max()) if len(imbalance) else np.nan
FIELD_STRENGTH_SUMMARY = "available" if scanner_fold_audit.magnetic_field_strength_median.notna().any() else "not available in baseline manifest"
print("largest manufacturer fraction spread across folds:", LARGEST_MANUFACTURER_IMBALANCE)
print("MagneticFieldStrength:", FIELD_STRENGTH_SUMMARY)
display(scanner_fold_audit.sort_values("manufacturer_fraction", ascending=False).head(20))


## 9. Soft-BCE training, diagnostic metrics, timing, and explicit loader cleanup


In [ ]:
def move_batch(batch):
    # Tensor construction happens only in the notebook process, after NumPy IPC has completed.
    for k in ["images", "slots", "positions", "mask", "targets"]:
        value = batch.get(k)
        if value is None:
            continue
        if isinstance(value, np.ndarray):
            value = torch.from_numpy(value)
        batch[k] = value.to(DEVICE, non_blocking=False)
    return batch

def build_model_and_optimizer(epochs):
    base = DenseSliceMIL().to(DEVICE)
    encoder_params = list(base.encoder.parameters()); encoder_ids = {id(p) for p in encoder_params}
    head_params = [p for p in base.parameters() if id(p) not in encoder_ids]
    optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": CONFIG["encoder_lr"]},
                                   {"params": head_params, "lr": CONFIG["head_lr"]}],
                                  weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    # DataParallel with one study cannot split useful work and still replicates the model.
    use_data_parallel = torch.cuda.device_count() > 1 and CONFIG["study_batch_size"] >= 2
    model = nn.DataParallel(base) if use_data_parallel else base
    print("DataParallel enabled:", use_data_parallel)
    return model, optimizer, scheduler

def train_one_epoch(model, loader, optimizer, scaler):
    model.train(); total, count = 0.0, 0
    accumulation = int(CONFIG["gradient_accumulation_steps"])
    optimizer.zero_grad(set_to_none=True)
    for step, batch in enumerate(loader, 1):
        batch = move_batch(batch)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            logits = model(batch["images"], batch["slots"], batch["positions"], batch["mask"])
            raw_loss = nn.functional.binary_cross_entropy_with_logits(logits, batch["targets"])
            loss = raw_loss / accumulation
        scaler.scale(loss).backward()
        should_step = step % accumulation == 0 or step == len(loader)
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip_norm"])
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        total += float(raw_loss.detach()) * len(batch["uids"]); count += len(batch["uids"])
        if step == 1:
            print_memory_stats("after first training batch")
    return total / max(1, count)

def per_target_metrics(y, p, soft=False):
    yy = (y >= 0.5).astype(np.uint8) if soft else y.astype(np.uint8)
    rows = []
    for j, label in enumerate(LABELS):
        if len(np.unique(yy[:, j])) < 2: auc = ap = np.nan
        else: auc, ap = roc_auc_score(yy[:, j], p[:, j]), average_precision_score(yy[:, j], p[:, j])
        rows.append({"target": label, "auc": auc, "ap": ap, "n_positive": int(yy[:, j].sum()), "n": len(yy)})
    return pd.DataFrame(rows)

@torch.inference_mode()
def predict(model, loader, return_attention=False):
    model.eval(); probs, targets, uids, attention_batches, metadata = [], [], [], [], []
    total_loss, count = 0.0, 0
    for batch in loader:
        raw_meta = ({k: np.array(batch[k], copy=True) for k in ["slots", "positions", "slice_indices", "mask"]}
                    if return_attention else None)
        batch = move_batch(batch)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            out = model(batch["images"], batch["slots"], batch["positions"], batch["mask"], return_attention=return_attention)
            logits, attn = out if return_attention else (out, None)
            if batch["targets"] is not None:
                loss = nn.functional.binary_cross_entropy_with_logits(logits, batch["targets"], reduction="sum")
                total_loss += float(loss); count += batch["targets"].numel()
        probs.append(torch.sigmoid(logits).cpu().numpy()); uids.extend(batch["uids"])
        if batch["targets"] is not None: targets.append(batch["targets"].cpu().numpy())
        if return_attention: attention_batches.append(attn.cpu().numpy()); metadata.append(raw_meta)
    return {"uids": uids, "prob": np.concatenate(probs),
            "target": np.concatenate(targets) if targets else None,
            "bce": total_loss / max(1, count), "attention": attention_batches, "metadata": metadata}

def save_checkpoint(path, model, extra):
    torch.save({"state_dict": unwrap(model).state_dict(), "config": CONFIG, "labels": LABELS,
                "slots": SLOTS, **extra}, path)

def shutdown_loader(loader):
    if loader is None: return
    try:
        iterator = getattr(loader, "_iterator", None)
        if iterator is not None and hasattr(iterator, "_shutdown_workers"): iterator._shutdown_workers()
    except Exception as exc:
        print("loader shutdown warning:", repr(exc))


## 10. Development CV on the first existing folds, with memory and runtime tracking


In [ ]:
history_rows, cv_rows, cv_target_rows = [], [], []
gold_fold_rows, gold_fold_target_rows, best_epochs, fold_runtime_minutes = [], [], [], []
weak_target_frame = weak_manifest[["StudyInstanceUID"] + LABELS]
gold_target_frame = gold[["StudyInstanceUID"] + LABELS]

for fold in ACTIVE_FOLDS:
    fold_start = monotonic(); seed_everything(CONFIG["seed"] + fold)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    print_memory_stats(f"before fold {fold}")
    tr_idx, va_idx = np.where(fold_ids != fold)[0], np.where(fold_ids == fold)[0]
    tr_ids = set(weak_manifest.iloc[tr_idx].StudyInstanceUID.astype(str)); va_ids = set(weak_manifest.iloc[va_idx].StudyInstanceUID.astype(str))
    assert tr_ids.isdisjoint(gold_ids) and va_ids.isdisjoint(gold_ids)
    assert set(weak_manifest.iloc[tr_idx].report_group.astype(str)).isdisjoint(
        set(weak_manifest.iloc[va_idx].report_group.astype(str)))
    tr_cache, va_cache = weak_cache_manifest.iloc[tr_idx], weak_cache_manifest.iloc[va_idx]
    tr_loader = make_loader(tr_cache, weak_target_frame, training=True, shuffle=True)
    va_loader = make_loader(va_cache, weak_target_frame, training=False, shuffle=False)
    gold_loader = make_loader(gold_cache_manifest, gold_target_frame, training=False, shuffle=False)
    print_memory_stats(f"after DataLoader creation fold {fold}")

    max_epochs = (1 if CONFIG["smoke_test"] else
                  CONFIG["preflight_epochs"] if CONFIG["preflight_only"] else CONFIG["max_epochs"])
    model, optimizer, scheduler = build_model_and_optimizer(max_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    best_bce, best_epoch, best_state, stale = np.inf, 0, None, 0
    clean_epoch_rss = []
    for epoch in range(1, max_epochs + 1):
        epoch_start = monotonic()
        train_bce = train_one_epoch(model, tr_loader, optimizer, scaler)
        val = predict(model, va_loader)
        diag = per_target_metrics(val["target"], val["prob"], soft=True)
        val_bce = float(val["bce"]); diag_auc = float(diag.auc.mean(skipna=True)); epoch_seconds = monotonic() - epoch_start
        history_rows.append({"stage": "cv", "fold": fold, "epoch": epoch, "train_bce": train_bce,
                             "val_bce": val_bce, "diagnostic_weak_macro_auc": diag_auc,
                             "epoch_seconds": epoch_seconds})
        pd.DataFrame(history_rows).to_csv(OUT / "training_history.csv", index=False)
        print(f"fold={fold} epoch={epoch} train_bce={train_bce:.6f} val_bce={val_bce:.6f} "
              f"diag_auc={diag_auc:.6f} sec/epoch={epoch_seconds:.1f}")
        if val_bce < best_bce - 1e-5:
            best_bce, best_epoch, stale = val_bce, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in unwrap(model).state_dict().items()}
        else:
            stale += 1
        scheduler.step()
        del val, diag
        release_host_memory()
        memory_row = print_memory_stats(f"fold {fold} after cleaned epoch {epoch}")
        clean_epoch_rss.append(float(memory_row["total_process_tree_rss_gib"]))
        if len(clean_epoch_rss) >= 2:
            retained_growth = clean_epoch_rss[-1] - clean_epoch_rss[-2]
            print(f"allocator-retained RSS growth this epoch: {retained_growth:.3f} GiB")
        low_available = (
            np.isfinite(memory_row["system_available_gib"])
            and memory_row["system_available_gib"] < CONFIG["min_system_available_gib"]
        )
        if low_available:
            if best_state is not None:
                torch.save({"state_dict": best_state, "config": CONFIG, "labels": LABELS, "slots": SLOTS,
                            "fold": fold, "best_epoch": best_epoch, "best_val_bce": best_bce,
                            "stopped_by_memory_guard": True}, OUT / f"fold_{fold}_memory_guard.pt")
            raise RuntimeError(
                f"HOST-RAM SAFETY GUARD: only {memory_row['system_available_gib']:.3f} GiB system RAM remains. "
                "Stopped before kernel OOM."
            )
        if stale >= CONFIG["patience"]: break

    if best_state is None: raise RuntimeError(f"Fold {fold} produced no valid checkpoint")
    unwrap(model).load_state_dict(best_state)
    save_checkpoint(OUT / f"fold_{fold}.pt", model, {"fold": fold, "best_epoch": best_epoch, "best_val_bce": best_bce})
    best_epochs.append(best_epoch)

    val = predict(model, va_loader); diag = per_target_metrics(val["target"], val["prob"], soft=True)
    for _, r in diag.iterrows(): cv_target_rows.append({"fold": fold, **r.to_dict()})
    cv_rows.append({"fold": fold, "best_epoch": best_epoch, "best_validation_bce": best_bce,
                    "diagnostic_weak_macro_auc": float(diag.auc.mean(skipna=True))})

    gold_start = monotonic(); ga = predict(model, gold_loader)
    gm = per_target_metrics(ga["target"], ga["prob"], soft=False); gold_seconds = monotonic() - gold_start
    gold_fold_rows.append({"fold": fold, "gold_macro_auc": float(gm.auc.mean(skipna=True)),
                           "gold_macro_ap": float(gm.ap.mean(skipna=True)), "gold_audit_seconds": gold_seconds})
    for _, r in gm.iterrows(): gold_fold_target_rows.append({"fold": fold, **r.to_dict()})
    print_memory_stats(f"after gold audit fold {fold}")

    fold_minutes = (monotonic() - fold_start) / 60; fold_runtime_minutes.append(fold_minutes)
    cv_rows[-1]["fold_runtime_minutes"] = fold_minutes
    projected = np.mean(fold_runtime_minutes) * 5 / 60
    print(f"fold {fold} elapsed minutes={fold_minutes:.1f}; projected 5-fold training hours={projected:.2f}")
    pd.DataFrame(cv_rows).to_csv(OUT / "cv_metrics.csv", index=False)
    pd.DataFrame(cv_target_rows).to_csv(OUT / "cv_per_target_metrics.csv", index=False)
    pd.DataFrame(gold_fold_rows).to_csv(OUT / "gold_fold_metrics.csv", index=False)
    pd.DataFrame(gold_fold_target_rows).to_csv(OUT / "gold_fold_per_target_metrics.csv", index=False)

    shutdown_loader(tr_loader); shutdown_loader(va_loader); shutdown_loader(gold_loader)
    del model, optimizer, scheduler, scaler, tr_loader, va_loader, gold_loader
    del val, diag, ga, gm, best_state, tr_cache, va_cache, tr_idx, va_idx, tr_ids, va_ids
    release_host_memory()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try: torch.cuda.ipc_collect()
        except Exception: pass
    print_memory_stats(f"after fold cleanup {fold}")

display(pd.DataFrame(cv_rows)); display(pd.DataFrame(gold_fold_rows))


## 11. Conditional final full-data training

The debug default (`DEV_NUM_FOLDS=2`) deliberately skips this stage. Set `dev_num_folds=5` only after the memory logs show stable cleanup across fold boundaries.


In [ ]:
FINAL_EPOCHS = None; final_model = None
if CONFIG["preflight_only"] or CONFIG["dev_num_folds"] < 5:
    print("FINAL FULL-DATA TRAINING SKIPPED IN PREFLIGHT/DEV MODE")
else:
    FINAL_EPOCHS = max(1, int(np.rint(np.median(best_epochs))))
    print("fold best epochs:", best_epochs, "FINAL_EPOCHS:", FINAL_EPOCHS)
    print_memory_stats("before final full-data training")
    seed_everything(CONFIG["seed"] + 100)
    all_loader = make_loader(weak_cache_manifest, weak_target_frame, training=True, shuffle=True)
    final_model, optimizer, scheduler = build_model_and_optimizer(FINAL_EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    for epoch in range(1, FINAL_EPOCHS + 1):
        t0 = monotonic(); train_bce = train_one_epoch(final_model, all_loader, optimizer, scaler); scheduler.step()
        history_rows.append({"stage": "final", "fold": -1, "epoch": epoch, "train_bce": train_bce,
                             "val_bce": np.nan, "diagnostic_weak_macro_auc": np.nan,
                             "epoch_seconds": monotonic() - t0})
        pd.DataFrame(history_rows).to_csv(OUT / "training_history.csv", index=False)
        print(f"final epoch={epoch}/{FINAL_EPOCHS} train_bce={train_bce:.6f}")
        print_memory_stats(f"final after epoch {epoch}")
    save_checkpoint(OUT / "05A_dense_mil_final.pt", final_model, {"final_epochs": FINAL_EPOCHS})
    shutdown_loader(all_loader); del all_loader, optimizer, scheduler, scaler
    gc.collect()


## 12. Conditional final-model gold attention audit

Fold gold audits are always produced. This extra final-model audit exists only in a full five-fold run that trains the deployment model.


In [ ]:
gold_final_summary = None; final_metrics = None
if final_model is None:
    print("Final-model gold audit skipped: no final full-data model in development mode")
else:
    gold_loader = make_loader(gold_cache_manifest, gold_target_frame, training=False, shuffle=False)
    final_gold = predict(final_model, gold_loader, return_attention=True)
    final_metrics = per_target_metrics(final_gold["target"], final_gold["prob"], soft=False)
    gold_final_summary = pd.DataFrame([{"gold_macro_auc": float(final_metrics.auc.mean(skipna=True)),
        "gold_macro_ap": float(final_metrics.ap.mean(skipna=True)), "n_gold": len(gold_manifest), "audit_only": True}])
    gold_final_summary.to_csv(OUT / "gold_final_metrics.csv", index=False)
    final_metrics.to_csv(OUT / "gold_final_per_target_metrics.csv", index=False)
    pred_df = pd.DataFrame({"StudyInstanceUID": final_gold["uids"]})
    for j, label in enumerate(LABELS):
        pred_df[f"pred__{label}"] = final_gold["prob"][:, j]; pred_df[f"truth__{label}"] = final_gold["target"][:, j]
    pred_df.to_csv(OUT / "gold_final_predictions.csv", index=False)

    attention_rows, uid_offset = [], 0
    for aw, meta in zip(final_gold["attention"], final_gold["metadata"]):
        mean_aw = aw.mean(axis=1)
        for bi in range(len(mean_aw)):
            uid = final_gold["uids"][uid_offset + bi]; valid = np.asarray(meta["mask"][bi], dtype=bool)
            for token in np.where(valid)[0]:
                slot_id, slice_idx = int(meta["slots"][bi, token]), int(meta["slice_indices"][bi, token])
                pos = float(meta["positions"][bi, token])
                for target_id, target in enumerate(LABELS):
                    attention_rows.append({"StudyInstanceUID": uid, "target": target, "slot": SLOTS[slot_id],
                                           "slice_index": slice_idx, "position_norm": pos,
                                           "attention_weight": float(mean_aw[bi, target_id, token])})
        uid_offset += len(mean_aw)
    attention_df = pd.DataFrame(attention_rows); attention_df.to_csv(OUT / "gold_attention_summary.csv", index=False)
    attention_df.groupby(["target", "slot"], as_index=False).agg(
        mean_attention=("attention_weight", "mean"), total_attention=("attention_weight", "sum"),
        n_tokens=("attention_weight", "size")).to_csv(OUT / "target_slot_attention.csv", index=False)
    shutdown_loader(gold_loader); del gold_loader, final_gold, attention_df, attention_rows
    del final_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print_memory_stats("after final gold audit cleanup")


## 13. Development report and compact artifact ZIP

The attached 11.9 GiB cache is referenced but never copied into the ZIP.


In [ ]:
CONFIG.update({"labels": LABELS, "competition_root": str(COMP_ROOT), "silver_path": str(PATHS["silver"]),
               "series_manifest_path": str(PATHS["manifest"]), "gpus": GPU_NAMES,
               "final_epochs": FINAL_EPOCHS, "model_parameters": PARAMS, "trainable_parameters": TRAINABLE_PARAMS})
with open(OUT / "config.json", "w", encoding="utf-8") as f: json.dump(CONFIG, f, indent=2)
memory_df = pd.DataFrame(MEMORY_HISTORY); memory_df.to_csv(OUT / "memory_history.csv", index=False)
cv_df, gf_df = pd.DataFrame(cv_rows), pd.DataFrame(gold_fold_rows)
peak_rss = float(memory_df.process_rss_gib.max())
peak_cuda_alloc = float(memory_df.cuda_max_allocated_gib.max())
peak_cuda_reserved = float(memory_df.cuda_reserved_gib.max())
projected_5fold_hours = float(np.mean(fold_runtime_minutes) * 5 / 60)
cleanup_rss = memory_df[memory_df.tag.str.startswith("after fold cleanup")].process_rss_gib.dropna().to_numpy()
cleanup_growth = float(cleanup_rss[-1] - cleanup_rss[0]) if len(cleanup_rss) >= 2 else np.inf
system_total = float(memory_df.system_total_gib.dropna().iloc[-1]) if memory_df.system_total_gib.notna().any() else np.nan
memory_safe = np.isfinite(system_total) and peak_rss < 0.80 * system_total and cleanup_growth <= 1.0
SAFE_FOR_FULL_RUN = "YES" if (len(cv_rows) == CONFIG["dev_num_folds"] and memory_safe and projected_5fold_hours <= 8.0) else "NO"
CACHE_REUSED = int(STATS["reused_local_cached_studies"] + STATS["reused_attached_cached_studies"])

lines = ["RSNA 05A DENSE-SLICE MIL — RAMFIX DEVELOPMENT RESULTS", "=" * 76,
         f"Run mode PREFLIGHT_ONLY: {CONFIG['preflight_only']}",
         f"Run mode ACTIVE_FOLDS: {CONFIG['active_folds']}",
         f"Final full-data training skipped: {CONFIG['preflight_only'] or CONFIG['dev_num_folds'] < 5}",
         f"Total runtime seconds: {monotonic() - RUN_START:.1f}", f"GPU(s): {GPU_NAMES}", "",
         "MEMORY", f"Peak process RSS GiB: {peak_rss:.3f}",
         f"Peak CUDA allocated GiB: {peak_cuda_alloc:.3f}", f"Peak CUDA reserved GiB: {peak_cuda_reserved:.3f}",
         f"Post-fold cleanup RSS growth GiB: {cleanup_growth:.3f}", "", "DATA",
         f"Weak train count: {len(weak_manifest)}", f"Gold audit count: {len(gold_manifest)}",
         f"Average valid slices/study: {weak_cache_manifest.n_tokens.mean():.3f}",
         f"Cache reused study count: {CACHE_REUSED}",
         f"Cache rebuilt study count: {int(STATS['rebuilt_cached_studies'])}",
         f"Referenced cache GiB: {CACHE_BYTES / 2**30:.3f}", "", "COMPLETED FOLDS"]
for f in ACTIVE_FOLDS:
    c = cv_df.loc[cv_df.fold.eq(f)].iloc[0]; g = gf_df.loc[gf_df.fold.eq(f)].iloc[0]
    lines.append(f"- fold {f}: best_epoch={int(c.best_epoch)}, best_val_soft_BCE={c.best_validation_bce:.6f}, "
                 f"diagnostic_pseudo_AUC={c.diagnostic_weak_macro_auc:.6f}, gold_macro_AUC={g.gold_macro_auc:.6f}, "
                 f"gold_macro_AP={g.gold_macro_ap:.6f}, runtime_minutes={c.fold_runtime_minutes:.1f}")
lines += ["", "SCANNER/SITE-LIKE AUDIT",
          f"Largest manufacturer fraction imbalance across folds: {LARGEST_MANUFACTURER_IMBALANCE:.6f}",
          f"MagneticFieldStrength distribution: {FIELD_STRENGTH_SUMMARY}", "",
          f"Projected 5-fold training runtime hours: {projected_5fold_hours:.3f}",
          f"SAFE_FOR_FULL_RUN = {SAFE_FOR_FULL_RUN}",
          "Execution strategy: one fold per fresh Kaggle kernel to bound allocator-retained RAM.",
          "No Kaggle leaderboard improvement is claimed from this development run."]
report = "\n".join(lines)
(OUT / "RESULTS_TO_SEND_BACK.txt").write_text(report, encoding="utf-8"); print(report)

required = ["config.json", "fold_assignments.csv", "training_history.csv", "cv_metrics.csv",
            "cv_per_target_metrics.csv", "gold_fold_metrics.csv", "gold_fold_per_target_metrics.csv",
            "memory_history.csv", "scanner_fold_audit.csv", "RESULTS_TO_SEND_BACK.txt"]
required += [f"fold_{i}.pt" for i in ACTIVE_FOLDS]
if not CONFIG["preflight_only"] and CONFIG["dev_num_folds"] == 5:
    required += ["05A_dense_mil_final.pt", "gold_final_metrics.csv", "gold_final_per_target_metrics.csv",
                 "gold_final_predictions.csv", "gold_attention_summary.csv", "target_slot_attention.csv"]
zip_path = Path("/kaggle/working/RSNA_05A_DenseMIL_Artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    for name in required:
        p = OUT / name
        if not p.exists(): raise FileNotFoundError(p)
        z.write(p, arcname=name)
print("Artifacts:", zip_path)
